## Ingest results.json

In [0]:
%run /Workspace/Users/nicolas97alonso@gmail.com/databricks-course/utils/vairables

### Step 1 - Ingest results.json

#### 1.1 Define the schema

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, FloatType,  StringType

In [0]:
results_schema = StructType([
    StructField("resultId", IntegerType(), False),
    StructField("raceId", IntegerType(), True),
    StructField("driverId", IntegerType(), False),
    StructField("constructorId", IntegerType(), False),
    StructField("number", IntegerType(), False),
    StructField("grid", IntegerType(), False),
    StructField("position", IntegerType(), False),
    StructField("positionText", StringType(), False),
    StructField("positionOrder", IntegerType(), False),
    StructField("points", FloatType(), False),
    StructField("laps", IntegerType(), False),
    StructField("time", StringType(), False),
    StructField("milliseconds", IntegerType(), False),
    StructField("fastestLap", IntegerType(), False),
    StructField("rank", IntegerType(), False),
    StructField("fastestLapTime", StringType(), False),
    StructField("fastestLapSpeed", FloatType(), False),
    StructField("statusId", StringType(), False)
])

#### 1.2 Read json file

In [0]:
raw_path = define_path("raw")

In [0]:
results_df = (
    spark.read
        .format("json")
        .schema(results_schema)
        .load(f"{raw_path}/results.json")
    )

### Step 2 - Rename Columns

In [0]:
results_renamed_df = (
    results_df
        .withColumnRenamed("resultId", "result_id")
        .withColumnRenamed("raceId", "race_id")
        .withColumnRenamed("driverId", "driver_id")
        .withColumnRenamed("constructorId", "constructor_id")
        .withColumnRenamed("positionText", "position_text")
        .withColumnRenamed("positionOrder", "position_order")
        .withColumnRenamed("fastestLapTime", "fastest_lap_time")
        .withColumnRenamed("fastestLapSpeed", "fastest_lap_speed")
)

In [0]:
from pyspark.sql.functions import col, current_timestamp

### Step 3 - Drop columns

In [0]:
results_dropped_df = results_renamed_df.drop(col("statusId"))

### Step 4 - Add column

In [0]:
results_final_df = results_dropped_df.withColumn("ingestion_date", current_timestamp())

### Step 5 - Write parquet file 

In [0]:
processed_path = define_path("processed")

In [0]:
results_final_df.write.format("parquet").mode("overwrite").save(f"{processed_path}/results")